In [1]:
import sqlite3
import plotly.express
import pandas
import plotly.graph_objects

# Connect to the SQLite database
db_path = '/home/dimitri/code/oll_onemax/computed/sunshine/default_sb3_ppo/_merged.db'
conn = sqlite3.connect(db_path)

# Load data from the EVALUATION_EPISODES table
query_evaluation = """
SELECT policy_id, AVG(num_function_evaluations) AS avg_function_evaluations, COUNT(*) AS row_count
FROM EVALUATION_EPISODES
GROUP BY policy_id
"""
df_evaluation = pandas.read_sql_query(query_evaluation, conn)
df_evaluation.set_index('policy_id', inplace=True)

# Load num_total_timesteps from the CONSTRUCTED_POLICIES table
query_policies = """
SELECT policy_id, num_total_timesteps
FROM CONSTRUCTED_POLICIES
"""
df_policies = pandas.read_sql_query(query_policies, conn)

# Merge the two dataframes on policy_id
df_merged = df_evaluation.merge(df_policies.drop_duplicates(subset='policy_id'), on='policy_id', how='left')
df_merged.set_index('policy_id', inplace=True)

# Close the connection
conn.close()

# Separate the baseline data (policy_id = -1)
df_baseline = df_merged[df_merged.index == -1]
df_others = df_merged[df_merged.index != -1]

# Create the line plot for other policies
fig = plotly.express.line(
    df_others,
    x='num_total_timesteps',
    y='avg_function_evaluations',
    labels={'num_total_timesteps': 'Number of Total Timesteps', 'avg_function_evaluations': 'Average Function Evaluations'},
    title='Average Function Evaluations by Number of Total Timesteps'
)

# Update the plot with hover data for other policies
fig.update_traces(
    mode='markers+lines',
    hovertemplate='<b>Number of Total Timesteps:</b> %{x}<br>' +
                  '<b>Average Function Evaluations:</b> %{y}<br>' +
                  '<b>Row Count:</b> %{customdata[0]}'
)

# Add hover data for other policies
fig.update_traces(customdata=df_others[['row_count']])

# Add the baseline line with hoverable points
if not df_baseline.empty:
    baseline_value = df_baseline['avg_function_evaluations'].values[0]
    fig.add_trace(
        plotly.graph_objects.Scatter(
            x=df_others['num_total_timesteps'],
            y=[baseline_value] * len(df_others),
            mode='lines',
            line=dict(color='orange', dash='dash'),
            name='Baseline',
            hoverinfo='y',
            hovertemplate='<b>Baseline:</b><br>' +
                          '<b>Average Function Evaluations:</b> %{y}'
        )
    )

# Update layout to start y-axis from 0
fig.update_layout(yaxis=dict(range=[0, None]))

# Show the plot
fig.show()

DatabaseError: Execution failed on sql '
SELECT policy_id, AVG(num_function_evaluations) AS avg_function_evaluations, COUNT(*) AS row_count
FROM EVALUATION_EPISODES
GROUP BY policy_id
': no such table: EVALUATION_EPISODES